# Task 7: Creating a RESTful API using Django Rest Framework

**Goal:** Develop a RESTful API using Django Rest Framework, define serializers, and test API endpoints using Postman.

In this notebook, we will create a simple **Student API**.

The API will support:
- `GET` all students
- `POST` create a new student
- `GET` one student by ID
- `PUT/PATCH` update a student
- `DELETE` remove a student

**Main tools used:**
- Django
- Django Rest Framework
- SQLite database
- Postman for endpoint testing

In [ ]:
# Install Django and Django Rest Framework
!pip install django djangorestframework

## Step 1: Create Django project files

This notebook creates the Django project manually using Python file writing.

In [ ]:
import os
from pathlib import Path

# Main project folder
BASE_DIR = Path("student_api_project")

# Create folders
(BASE_DIR / "student_api").mkdir(parents=True, exist_ok=True)
(BASE_DIR / "api").mkdir(parents=True, exist_ok=True)

# Create __init__.py files so Python treats folders as packages
(BASE_DIR / "student_api" / "__init__.py").write_text("")
(BASE_DIR / "api" / "__init__.py").write_text("")

print("Project folders created successfully.")

## Step 2: Create Django settings file

In [ ]:
settings_code = """
from pathlib import Path

BASE_DIR = Path(__file__).resolve().parent.parent

SECRET_KEY = "dev-secret-key-for-student-api"

DEBUG = True

# Allow all hosts for development/testing.
# In production, replace this with your real domain.
ALLOWED_HOSTS = ["*"]

INSTALLED_APPS = [
    "django.contrib.admin",
    "django.contrib.auth",
    "django.contrib.contenttypes",
    "django.contrib.sessions",
    "django.contrib.messages",
    "django.contrib.staticfiles",

    # Third-party app
    "rest_framework",

    # Our custom app
    "api",
]

MIDDLEWARE = [
    "django.middleware.security.SecurityMiddleware",
    "django.contrib.sessions.middleware.SessionMiddleware",
    "django.middleware.common.CommonMiddleware",
    "django.middleware.csrf.CsrfViewMiddleware",
    "django.contrib.auth.middleware.AuthenticationMiddleware",
    "django.contrib.messages.middleware.MessageMiddleware",
    "django.middleware.clickjacking.XFrameOptionsMiddleware",
]

ROOT_URLCONF = "student_api.urls"

TEMPLATES = [
    {
        "BACKEND": "django.template.backends.django.DjangoTemplates",
        "DIRS": [],
        "APP_DIRS": True,
        "OPTIONS": {
            "context_processors": [
                "django.template.context_processors.debug",
                "django.template.context_processors.request",
                "django.contrib.auth.context_processors.auth",
                "django.contrib.messages.context_processors.messages",
            ],
        },
    },
]

WSGI_APPLICATION = "student_api.wsgi.application"

DATABASES = {
    "default": {
        "ENGINE": "django.db.backends.sqlite3",
        "NAME": BASE_DIR / "db.sqlite3",
    }
}

LANGUAGE_CODE = "en-us"
TIME_ZONE = "Asia/Karachi"
USE_I18N = True
USE_TZ = True

STATIC_URL = "static/"

DEFAULT_AUTO_FIELD = "django.db.models.BigAutoField"

REST_FRAMEWORK = {
    "DEFAULT_RENDERER_CLASSES": [
        "rest_framework.renderers.JSONRenderer",
        "rest_framework.renderers.BrowsableAPIRenderer",
    ]
}
"""

(BASE_DIR / "student_api" / "settings.py").write_text(settings_code)
print("settings.py created successfully.")

## Step 3: Create project URL and WSGI files

In [ ]:
urls_code = """
from django.contrib import admin
from django.urls import path, include
from rest_framework.routers import DefaultRouter
from api.views import StudentViewSet

# Router automatically creates RESTful API URLs
router = DefaultRouter()
router.register(r"students", StudentViewSet, basename="student")

urlpatterns = [
    path("admin/", admin.site.urls),
    path("api/", include(router.urls)),
]
"""

wsgi_code = """
import os
from django.core.wsgi import get_wsgi_application

os.environ.setdefault("DJANGO_SETTINGS_MODULE", "student_api.settings")

application = get_wsgi_application()
"""

(BASE_DIR / "student_api" / "urls.py").write_text(urls_code)
(BASE_DIR / "student_api" / "wsgi.py").write_text(wsgi_code)

print("urls.py and wsgi.py created successfully.")

## Step 4: Create Django app files

We will define a `Student` model.

In [ ]:
apps_code = """
from django.apps import AppConfig

class ApiConfig(AppConfig):
    default_auto_field = "django.db.models.BigAutoField"
    name = "api"
"""

models_code = """
from django.db import models

class Student(models.Model):
    # Student name
    name = models.CharField(max_length=100)

    # Student email
    email = models.EmailField(unique=True)

    # Department name
    department = models.CharField(max_length=100)

    # Semester number
    semester = models.IntegerField()

    # Student CGPA
    cgpa = models.FloatField()

    # This method controls how the object appears in admin panel
    def __str__(self):
        return self.name
"""

admin_code = """
from django.contrib import admin
from .models import Student

# Register Student model in admin panel
admin.site.register(Student)
"""

(BASE_DIR / "api" / "apps.py").write_text(apps_code)
(BASE_DIR / "api" / "models.py").write_text(models_code)
(BASE_DIR / "api" / "admin.py").write_text(admin_code)

print("App files created successfully.")

## Step 5: Create serializer

Serializer converts Django model objects into JSON and also validates incoming JSON data.

In [ ]:
serializers_code = """
from rest_framework import serializers
from .models import Student

class StudentSerializer(serializers.ModelSerializer):
    class Meta:
        model = Student

        # These fields will be visible in API JSON response
        fields = ["id", "name", "email", "department", "semester", "cgpa"]

    def validate_cgpa(self, value):
        # Custom validation: CGPA must be between 0 and 4
        if value < 0 or value > 4:
            raise serializers.ValidationError("CGPA must be between 0 and 4.")
        return value

    def validate_semester(self, value):
        # Custom validation: semester must be positive
        if value <= 0:
            raise serializers.ValidationError("Semester must be greater than 0.")
        return value
"""

(BASE_DIR / "api" / "serializers.py").write_text(serializers_code)
print("serializers.py created successfully.")

## Step 6: Create API views

`ModelViewSet` automatically provides list, create, retrieve, update, and delete operations.

In [ ]:
views_code = """
from rest_framework import viewsets
from .models import Student
from .serializers import StudentSerializer

class StudentViewSet(viewsets.ModelViewSet):
    # Queryset tells DRF which data to use
    queryset = Student.objects.all().order_by("id")

    # Serializer tells DRF how to convert data to/from JSON
    serializer_class = StudentSerializer
"""

(BASE_DIR / "api" / "views.py").write_text(views_code)
print("views.py created successfully.")

## Step 7: Create manage.py

In [ ]:
manage_code = """#!/usr/bin/env python
import os
import sys

def main():
    os.environ.setdefault("DJANGO_SETTINGS_MODULE", "student_api.settings")
    try:
        from django.core.management import execute_from_command_line
    except ImportError as exc:
        raise ImportError("Could not import Django.") from exc
    execute_from_command_line(sys.argv)

if __name__ == "__main__":
    main()
"""

(BASE_DIR / "manage.py").write_text(manage_code)
print("manage.py created successfully.")

## Step 8: Make migrations and migrate database

In [ ]:
# Move into project folder and create database tables
%cd student_api_project

!python manage.py makemigrations
!python manage.py migrate

## Step 9: Add sample records using Django shell

In [ ]:
# This creates sample student records.
# If records already exist, it will not duplicate them.
!python manage.py shell -c "from api.models import Student; Student.objects.get_or_create(name='Ali Khan', email='ali@example.com', department='Computer Science', semester=6, cgpa=3.45); Student.objects.get_or_create(name='Ayesha Ahmed', email='ayesha@example.com', department='Data Science', semester=4, cgpa=3.72); Student.objects.get_or_create(name='Usman Raza', email='usman@example.com', department='Software Engineering', semester=7, cgpa=3.10); print('Sample records inserted successfully')"

## Step 10: Run Django development server

Run this cell and open:

`http://127.0.0.1:8000/api/students/`

You can also test the same URL in Postman.

In [ ]:
# Run the API server.
# Stop the cell when you are finished testing.
!python manage.py runserver 0.0.0.0:8000

## Postman Testing Guide

Use these endpoints in Postman:

### 1. Get all students

**Method:** `GET`  
**URL:** `http://127.0.0.1:8000/api/students/`

### 2. Get one student

**Method:** `GET`  
**URL:** `http://127.0.0.1:8000/api/students/1/`

### 3. Create a new student

**Method:** `POST`  
**URL:** `http://127.0.0.1:8000/api/students/`  
**Body type:** raw JSON

```json
{
  "name": "Sara Malik",
  "email": "sara@example.com",
  "department": "Artificial Intelligence",
  "semester": 5,
  "cgpa": 3.88
}
```

### 4. Update a student

**Method:** `PUT`  
**URL:** `http://127.0.0.1:8000/api/students/1/`

```json
{
  "name": "Ali Khan Updated",
  "email": "ali@example.com",
  "department": "Computer Science",
  "semester": 7,
  "cgpa": 3.60
}
```

### 5. Delete a student

**Method:** `DELETE`  
**URL:** `http://127.0.0.1:8000/api/students/1/`

This completes the Django Rest Framework API task.